# Case: Model Operationalization
### Part 1b: Deploying a model into production

Welcome to the production environment!

Once a month, we receive a dataset with new data on the employees and we want to alert the
HR department if some of them are in potential risk of leaving. The goal of this part is to
understand the required process and checks that are important to implement in production. Carefully
review the new data and transform it in a correct way - so that we can generate predictions for it.

Make sure all the files you need for the case are in the same directory as the python
scripts/notebooks.

Good luck and have fun!

In [1]:
# Imports
import pandas as pd
import random
import warnings
import sklearn
import pickle
warnings.filterwarnings("ignore")
pd.options.mode.chained_assignment = None

# We set a random seed to get the same results with every run
random.seed(15)

We have just received new data from HR, in the file `HR_new_recruitment.csv`. Do we see changes in the production (near real-time) data compared to the training (well governed) data?

In [2]:
# Put your path to the CSV file here
csv_path = 'HR_new_recruitment.csv'

In [12]:
# Load the new file and review some statistics
df = pd.read_csv(csv_path, header=0, sep=',')
df.describe()

,satisfaction_level,time_spent_company,number_project,average_monthly_hours,Work_accident,promotion_last_5years
count,734.000000,1119.000000,1119.000000,1119.000000,1119.000000,1119.000000
mean,0.534428,4.096515,3.804290,212.633601,0.088472,0.042895
std,0.268260,1.657524,1.619464,298.931038,0.284107,0.202712
min,0.090000,2.000000,2.000000,98.000000,0.000000,0.000000
25%,0.370000,3.000000,2.000000,148.000000,0.000000,0.000000
50%,0.500000,4.000000,4.000000,208.000000,0.000000,0.000000
75%,0.780000,5.000000,5.000000,255.000000,0.000000,0.000000
max,0.990000,10.000000,7.000000,10018.000000,1.000000,1.000000


In [4]:
df.sample(5)

,satisfaction_level,time_spent_company,last_evaluation,number_project,salary,average_monthly_hours,Work_accident,promotion_last_5years,department
976,NaN,3,54%,2,medium,141,0,0,technical
511,0.40,3,47%,2,medium,146,0,0,sales
44,0.63,3,98%,4,high,187,0,0,management
1104,0.40,3,56%,2,medium,148,0,0,technical
1069,NaN,3,57%,2,low,147,1,0,management


Most fields look similar to the training data. However, `last_evaluation` is a string, but it should be an int. We will have to transform that.

## Exercise: Load the relevant model objects that we pickled in the previous exercise

In [5]:
# Load the model
forest = pickle.load(open('model.pickle', 'rb'))

# Load the column names
col_names = pickle.load(open('columns.pickle', 'rb'))

# Load the imputation mean
satisfaction_level_mean = pickle.load(open('satisfaction_mean.pickle', 'rb'))

# Load the scaler
avg_monthly_hours_scaler = pickle.load(open('scaling.pickle', 'rb'))

# Load the salary encoder
salary_encoding = pickle.load(open('salary_encoding.pickle', 'rb'))

## Exercise: Create a pipeline to transform the new dataset

In [6]:
# Hint - use str.replace('%','') to remove specific punctuation from a string
def prediction_transformation(prediction_set, avg_monthly_hours_scaler, satisfaction_level_mean, salary_encoding, col_names):
    prediction_set_copy = prediction_set.copy()
    prediction_set_copy['department'] = prediction_set_copy['department'].apply(lambda x: 'other' if x not in ['RandD','management'] else x)
    prediction_set_copy = pd.concat([prediction_set_copy, pd.get_dummies(prediction_set_copy['department'])], axis=1)
    prediction_set_copy.drop('department', axis=1, inplace=True)
    prediction_set_copy.replace(salary_encoding, inplace=True)
    prediction_set_copy['hours_per_project'] = prediction_set_copy['average_monthly_hours'] / prediction_set_copy['number_project']
    prediction_set_copy['average_monthly_hours'] = avg_monthly_hours_scaler.transform(prediction_set_copy['average_monthly_hours'].values.reshape(-1,1))
    prediction_set_copy.satisfaction_level.fillna(satisfaction_level_mean, inplace=True)

    prediction_set_copy['last_evaluation'] = prediction_set.last_evaluation.str.replace('%', '').astype(int) * 0.01

    return prediction_set_copy.loc[:, col_names]

In [7]:
# Apply transformation
X_predict = prediction_transformation(df, avg_monthly_hours_scaler, satisfaction_level_mean, salary_encoding, col_names)

In [8]:
# Make new predictions
predicted = forest.predict(X_predict)

In [9]:
# Save the predictions to a csv file
X_predict['predictions'] = predicted
X_predict.to_csv('predictions.csv')

## Extra exercise: pickling transformations
What happens if we do not pickle our transformations but we do recompute them?

Try to figure it out using the following code. This is a test to illustrate what happens if you don't scale using the trained model, but create a new scaler instead.

In [10]:
trained_scaler_value = avg_monthly_hours_scaler.data_range_[0]
reshaped_column = df.average_monthly_hours.values.reshape(-1, 1)
new_data_scaler = sklearn.preprocessing.MinMaxScaler().fit(reshaped_column)
new_data_scaler_value = new_data_scaler.data_range_[0]
print(f'Trained model Max-Min range: {trained_scaler_value:.2f}')
print(f'New data Max-Min range: {new_data_scaler_value:.2f}')
print('This will cause incorrect predictions for sure')

Trained model Max-Min range: 214.00
New data Max-Min range: 9920.00
This will cause incorrect predictions for sure


Can you do the same for the impute?

In [11]:
print(f'Trained model imputation value: {satisfaction_level_mean:.2f}')
print(f'New data imputation value: {df.satisfaction_level.mean():.2f}')
print('This will cause incorrect predictions for sure')

Trained model imputation value: 0.62
New data imputation value: 0.53
This will cause incorrect predictions for sure


## Extra exercise: other transformations
Can you further improve the model by doing other transformations? For example, creating new variables?

If so, make sure that you pickle those as well!